In [1]:
import joblib
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:
original_df = pd.read_csv("../Data/CICIDS2017_combined.csv")
feature_names = original_df.drop(" Label", axis=1).columns

In [3]:
X_train_scaled = np.load('../Data/X_train_scaled.npy')
X_test_scaled = np.load('../Data/X_test_scaled.npy')

X_train = np.load('../Data/X_train.npy')
X_test = np.load('../Data/X_test.npy')

y_train = np.load('../Data/y_train.npy')
y_test = np.load('../Data/y_test.npy')


print("Train shape:", X_train_scaled.shape)
print("Test shape:", X_test_scaled.shape)

y_train_bin = (y_train != 0).astype(int)
print(pd.Series(y_train_bin).value_counts())

y_test_bin = (y_test != 0).astype(int)
print(pd.Series(y_test_bin).value_counts())


Train shape: (2262300, 78)
Test shape: (565576, 78)
0    1817055
1     445245
Name: count, dtype: int64
0    454265
1    111311
Name: count, dtype: int64


In [4]:
X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=feature_names
)

In [5]:
rf = joblib.load('../Data/Sampled_Data/rf.pkl')


In [8]:
from sklearn.model_selection import train_test_split

X_sample, _, y_sample, _ = train_test_split(
    X_test,
    y_test,
    test_size=0.9,
    stratify=y_test,
    random_state=42
)

X_sample = pd.DataFrame(
    X_sample,
    columns=feature_names
)

In [ ]:
X_sample_shap = X_sample.sample(3000, random_state=42)

explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_sample_shap)

In [17]:
print(shap_values.shape)
if isinstance(shap_values, list) == 3:
    shap_values = shap_values[:, :, 1]

print(shap_values.shape)

(3000, 78, 2)
(3000, 78, 2)


In [ ]:
mean_abs = np.mean(np.abs(shap_values), axis=0)

rf_importance = pd.DataFrame({
    "feature": X_sample_shap.columns,
    "importance": mean_abs
}).sort_values("importance", ascending=False)